# 06. Динамическая кластеризация по месяцам

Теперь переносим выбранный принцип построения сети на весь период 2023-01 – 2024-12. Для каждого месяца строится собственная экономическая сеть и выполняется Louvain.

In [1]:
import pandas as pd
import numpy as np
import networkx as nx

from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score
from sklearn.metrics import calinski_harabasz_score

## 1. Загрузка панели

Используем `features_basic`, подготовленный в предыдущих шагах.

In [2]:
features = pd.read_parquet(
    "../data/processed/features_basic.parquet"
)

features["date"] = pd.to_datetime(features["date"])

model_features = [
    "food_share",
    "health_share",
    "marketplaces_share",
    "catering_share",
    "transport_share"
]

months = sorted(features["date"].unique())

print("Количество месяцев:", len(months))
print("Первый месяц:", months[0])
print("Последний месяц:", months[-1])

Количество месяцев: 24
Первый месяц: 2023-01-01 00:00:00
Последний месяц: 2024-12-01 00:00:00


## 2. Фиксация параметров динамического прогона

Здесь сохраняется историческая экспериментальная конфигурация `resolution = 1.5`. Финальная версия проекта позднее пересчитывается в `14_final_model_125` с `resolution = 1.25`.

In [3]:
k = 10
resolution = 1.5
seed = 42

## 3. Функция построения месячной сети

Выносим повторяющуюся логику построения kNN-сети и RBF-весов в отдельную функцию.

In [4]:
def build_month_network(
    month_data,
    model_features,
    k=10,
    resolution=1.5,
    seed=42
):
    X = month_data[model_features].to_numpy()
    municipalities = month_data["territory_id"].to_numpy()

    actual_k = min(k, len(month_data) - 1)

    knn = NearestNeighbors(
        n_neighbors=actual_k + 1,
        metric="euclidean"
    )

    knn.fit(X)

    distances, indices = knn.kneighbors(X)

    neighbor_distances = distances[:, 1:]
    neighbor_indices = indices[:, 1:]

    sigma = np.median(neighbor_distances)

    edge_weights = np.exp(
        -neighbor_distances / sigma
    )

    edges = []

    for i in range(len(municipalities)):
        for j in range(actual_k):
            neighbor_idx = neighbor_indices[i, j]

            edges.append({
                "source": int(municipalities[i]),
                "target": int(municipalities[neighbor_idx]),
                "weight": float(edge_weights[i, j])
            })

    edges = pd.DataFrame(edges)

    edges["node_min"] = edges[["source", "target"]].min(axis=1)
    edges["node_max"] = edges[["source", "target"]].max(axis=1)

    edges_undirected = (
        edges
        .groupby(
            ["node_min", "node_max"],
            as_index=False
        )["weight"]
        .mean()
        .rename(
            columns={
                "node_min": "source",
                "node_max": "target"
            }
        )
    )

    G = nx.Graph()

    G.add_nodes_from(municipalities)

    for row in edges_undirected.itertuples(index=False):
        G.add_edge(
            int(row.source),
            int(row.target),
            weight=float(row.weight)
        )

    communities = nx.community.louvain_communities(
        G,
        weight="weight",
        resolution=resolution,
        seed=seed
    )

    territory_to_position = {
        territory_id: i
        for i, territory_id
        in enumerate(municipalities)
    }

    labels = np.full(
        len(municipalities),
        -1,
        dtype=int
    )

    for cluster_id, community in enumerate(communities):
        for territory_id in community:
            position = territory_to_position[territory_id]
            labels[position] = cluster_id

    modularity = nx.community.modularity(
        G,
        communities,
        weight="weight"
    )

    silhouette = silhouette_score(
        X,
        labels,
        metric="euclidean"
    )

    calinski = calinski_harabasz_score(
        X,
        labels
    )

    return {
        "edges": edges_undirected,
        "labels": labels,
        "communities": communities,
        "graph": G,
        "sigma": sigma,
        "modularity": modularity,
        "silhouette": silhouette,
        "calinski_harabasz": calinski
    }

## 4. Контрольный запуск на одном месяце

Перед массовым прогоном проверяем, что функция корректно работает на декабре 2024.

In [5]:
test_date = pd.Timestamp("2024-12-01")

test_data = (
    features[features["date"] == test_date]
    .copy()
)

test_result = build_month_network(
    test_data,
    model_features,
    k=k,
    resolution=resolution,
    seed=seed
)

print("Муниципалитетов:", len(test_data))
print("Рёбер:", len(test_result["edges"]))
print(
    "Кластеров:",
    len(test_result["communities"])
)
print(
    "Modularity:",
    test_result["modularity"]
)
print(
    "Silhouette:",
    test_result["silhouette"]
)
print(
    "Calinski-Harabasz:",
    test_result["calinski_harabasz"]
)

Муниципалитетов: 2103
Рёбер: 13757
Кластеров: 19
Modularity: 0.8139722519099776
Silhouette: 0.17842438742691852
Calinski-Harabasz: 914.5988331398605


## 5. Динамический прогон за 24 месяца

Для каждого месяца сохраняем кластер муниципалитета и ребра его экономической сети. Так формируется динамическая атрибутированная сеть.

In [6]:
all_cluster_results = []
all_edge_results = []

for month in months:

    month_data = (
        features[features["date"] == month]
        .copy()
    )

    result = build_month_network(
        month_data,
        model_features,
        k=k,
        resolution=resolution,
        seed=seed
    )

    month_clusters = month_data[
        ["territory_id"]
    ].copy()

    month_clusters["date"] = month
    month_clusters["cluster"] = result["labels"]

    all_cluster_results.append(
        month_clusters
    )

    month_edges = result["edges"].copy()
    month_edges["date"] = month

    all_edge_results.append(
        month_edges
    )

    print(
        month.strftime("%Y-%m"),
        "| МО:", len(month_data),
        "| кластеров:", len(result["communities"]),
        "| рёбер:", len(result["edges"]),
        "| silhouette:",
        round(result["silhouette"], 4)
    )

2023-01 | МО: 2143 | кластеров: 23 | рёбер: 14016 | silhouette: 0.0873
2023-02 | МО: 2141 | кластеров: 22 | рёбер: 14100 | silhouette: 0.1179
2023-03 | МО: 2149 | кластеров: 20 | рёбер: 14065 | silhouette: 0.1327
2023-04 | МО: 2129 | кластеров: 22 | рёбер: 13975 | silhouette: 0.1209
2023-05 | МО: 2121 | кластеров: 25 | рёбер: 13926 | silhouette: 0.1293
2023-06 | МО: 2119 | кластеров: 23 | рёбер: 13927 | silhouette: 0.1503
2023-07 | МО: 2126 | кластеров: 21 | рёбер: 13966 | silhouette: 0.1543
2023-08 | МО: 2118 | кластеров: 22 | рёбер: 13859 | silhouette: 0.1658
2023-09 | МО: 2109 | кластеров: 23 | рёбер: 13888 | silhouette: 0.1408
2023-10 | МО: 2103 | кластеров: 21 | рёбер: 13754 | silhouette: 0.1734
2023-11 | МО: 2093 | кластеров: 20 | рёбер: 13738 | silhouette: 0.1481
2023-12 | МО: 2103 | кластеров: 24 | рёбер: 13772 | silhouette: 0.1318
2024-01 | МО: 2072 | кластеров: 21 | рёбер: 13600 | silhouette: 0.1285
2024-02 | МО: 2078 | кластеров: 20 | рёбер: 13656 | silhouette: 0.1529
2024-0

## 6. Сбор результатов в единую панель

Объединяем месячные результаты в таблицы, которые станут основой межвременного анализа.

In [7]:
clusters_all = pd.concat(
    all_cluster_results,
    ignore_index=True
)

edges_all = pd.concat(
    all_edge_results,
    ignore_index=True
)

print(
    "Размер таблицы кластеров:",
    clusters_all.shape
)

print(
    "Размер таблицы рёбер:",
    edges_all.shape
)

Размер таблицы кластеров: (50521, 3)
Размер таблицы рёбер: (331489, 4)


## 7. Сохранение динамической сети

Сохраняем кластеризации и ребра всех месяцев для следующих ноутбуков.

In [8]:
clusters_all.to_parquet(
    "../data/processed/clusters_all_months.parquet",
    index=False
)

edges_all.to_parquet(
    "../data/processed/edges_all_months.parquet",
    index=False
)

print("Результаты сохранены.")

Результаты сохранены.
